# SRGD Real-ESRGAN Kaggle Notebook (T4 x2)

这个 notebook 把仓库的命令行流程改成 Kaggle 可直接运行的版本，目标环境是 **GPU T4 x2**。

Kaggle 运行前请在右侧设置里确认：

- Accelerator: `GPU T4 x2`
- Internet: `On`
- Add data: 加入 SRGD / Super Resolution in Video Games 数据集

默认 `RUN_MODE = "smoke"`，会先用少量样本和很短迭代检查整条链路。确认没问题后，把它改成 `"full"` 再跑完整实验。

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
import textwrap

KAGGLE = Path('/kaggle').exists()
WORK_ROOT = Path('/kaggle/working') if KAGGLE else Path.cwd()
INPUT_ROOT = Path('/kaggle/input') if KAGGLE else Path.cwd() / 'data'

PROJECT_REPO_URL = 'https://github.com/scwx-yxh/deeplearning_project.git'
REAL_ESRGAN_REPO_URL = 'https://github.com/xinntao/Real-ESRGAN.git'

PROJECT_DIR = WORK_ROOT / 'deeplearning_project'
REAL_ESRGAN_DIR = WORK_ROOT / 'Real-ESRGAN'
OUTPUT_DIR = WORK_ROOT / 'outputs'
DATA_DIR = PROJECT_DIR / 'data'
CONFIG_DIR = PROJECT_DIR / 'configs'

RUN_MODE = 'smoke'  # smoke | full
RUN_FINETUNE = True
RUN_PRETRAINED_INFERENCE = True
RUN_TEST_SUBMISSION = False

COURSE_EVAL_LIMIT = 100
PAIR_LIMIT = 64 if RUN_MODE == 'smoke' else COURSE_EVAL_LIMIT
INFERENCE_LIMIT = 16 if RUN_MODE == 'smoke' else COURSE_EVAL_LIMIT
GRID_LIMIT = 6 if RUN_MODE == 'smoke' else 12
TOTAL_ITER = 100 if RUN_MODE == 'smoke' else 5_000
SAVE_FREQ = 100 if RUN_MODE == 'smoke' else 1_000
GT_SIZE = 128 if RUN_MODE == 'smoke' else 256
BATCH_SIZE_PER_GPU = 2  # T4 16GB 通常 1-4 都可试；OOM 时改成 1
NUM_WORKERS_PER_GPU = 2
TILE = 256
MODEL_NAME = 'RealESRGAN_x4plus'

# 如果自动探测失败，可以手动填这些路径。
DATA_ROOT_OVERRIDE = None
LR_DIR_OVERRIDE = None
HR_DIR_OVERRIDE = None
TEST_LR_DIR_OVERRIDE = None
SAMPLE_SUBMISSION_OVERRIDE = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print('\n$ ' + ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

## 1. Clone repos and install dependencies

Kaggle 每次启动都是干净环境，所以这里会自动 clone 本项目和官方 Real-ESRGAN，并安装依赖。若你把本仓库作为 Kaggle dataset 上传，也可以直接把 `PROJECT_DIR` 改到对应目录。

In [ ]:
if not (PROJECT_DIR / 'requirements.txt').exists():
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run(['git', 'clone', PROJECT_REPO_URL, PROJECT_DIR])
else:
    print(f'Using existing project repo: {PROJECT_DIR}')

if not (REAL_ESRGAN_DIR / 'requirements.txt').exists():
    if REAL_ESRGAN_DIR.exists():
        shutil.rmtree(REAL_ESRGAN_DIR)
    run(['git', 'clone', '--depth', '1', REAL_ESRGAN_REPO_URL, REAL_ESRGAN_DIR])
else:
    print(f'Using existing Real-ESRGAN repo: {REAL_ESRGAN_DIR}')

DATA_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle generally already has torch/torchvision. Avoid forcing a CUDA stack reinstall.
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'wheel', 'cython', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', PROJECT_DIR / 'requirements.txt'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'basicsr>=1.4.2', 'facexlib>=0.2.5', 'gfpgan>=1.3.5'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REAL_ESRGAN_DIR / 'requirements.txt'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REAL_ESRGAN_DIR])

sys.path.insert(0, str(PROJECT_DIR / 'src'))

## 1.1. Apply notebook compatibility helpers

If this notebook clones the original GitHub repo, this cell injects the small Kaggle-specific helper updates used below.


In [ ]:
# Keep the notebook self-contained even when it clones the upstream repo.
(PROJECT_DIR / 'scripts' / 'run_realesrgan.py').write_text('from __future__ import annotations\n\nimport argparse\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nfrom PIL import Image\n\ntry:\n    from tqdm import tqdm\nexcept ImportError:\n    def tqdm(iterable, **_: object):\n        return iterable\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT / "src"))\n\nfrom srgd_realesrgan.paths import IMAGE_EXTENSIONS, ensure_dir, read_pairs_csv\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Run official Real-ESRGAN inference on paired LR images.")\n    parser.add_argument("--realesrgan-dir", type=Path, required=True, help="Path to cloned xinntao/Real-ESRGAN repo.")\n    parser.add_argument("--pairs", type=Path, required=True, help="Pairs CSV from prepare_pairs.py.")\n    parser.add_argument("--out-dir", type=Path, required=True, help="Folder for normalized SR PNG outputs.")\n    parser.add_argument("--model-name", default="RealESRGAN_x4plus", help="Real-ESRGAN model name.")\n    parser.add_argument("--model-path", type=Path, default=None, help="Optional custom Real-ESRGAN checkpoint.")\n    parser.add_argument("--outscale", type=float, default=4.0, help="Output scale passed to inference_realesrgan.py.")\n    parser.add_argument("--tile", type=int, default=0, help="Tile size. Use 128/256 on small GPUs.")\n    parser.add_argument("--gpu-id", type=int, default=None, help="Optional GPU id passed to inference_realesrgan.py.")\n    parser.add_argument("--ext", default="png", help="Output image extension passed to inference_realesrgan.py.")\n    parser.add_argument("--fp32", action="store_true", help="Use fp32 inference instead of half precision.")\n    parser.add_argument("--face-enhance", action="store_true", help="Enable GFPGAN face enhancement.")\n    parser.add_argument("--split", default=None, help="Optional split filter: train, val, or test.")\n    parser.add_argument("--limit", type=int, default=None, help="Optional maximum number of images.")\n    parser.add_argument("--shuffle", action="store_true", help="Shuffle before applying --limit.")\n    parser.add_argument("--seed", type=int, default=0, help="Random seed for --shuffle.")\n    return parser.parse_args()\n\n\ndef normalize_outputs(raw_dir: Path, out_dir: Path, rows: list[dict[str, str]], suffix: str) -> None:\n    for row in tqdm(rows, desc="Normalize outputs"):\n        pair_id = row["pair_id"]\n        matches = []\n        for ext in IMAGE_EXTENSIONS:\n            matches.extend(raw_dir.glob(f"{pair_id}_{suffix}{ext}"))\n            matches.extend(raw_dir.glob(f"{pair_id}_{suffix}{ext.upper()}"))\n        if not matches:\n            matches = list(raw_dir.glob(f"{pair_id}_*"))\n        if not matches:\n            raise FileNotFoundError(f"Real-ESRGAN output not found for pair_id={pair_id} in {raw_dir}")\n\n        image = Image.open(matches[0]).convert("RGB")\n        image.save(out_dir / f"{pair_id}.png")\n\n\ndef main() -> None:\n    args = parse_args()\n    realesrgan_dir = args.realesrgan_dir.resolve()\n    inference_script = realesrgan_dir / "inference_realesrgan.py"\n    if not inference_script.exists():\n        raise FileNotFoundError(f"Cannot find {inference_script}. Did you clone the official Real-ESRGAN repo?")\n\n    rows = read_pairs_csv(args.pairs, split=args.split, limit=args.limit, shuffle=args.shuffle, seed=args.seed)\n    out_dir = ensure_dir(args.out_dir)\n    input_dir = ensure_dir(out_dir / "_lr_inputs")\n    raw_dir = ensure_dir(out_dir / "_raw_realesrgan")\n    suffix = "sr"\n\n    for row in tqdm(rows, desc="Stage LR inputs"):\n        src = Path(row["lr_path"])\n        dst = input_dir / f"{row[\'pair_id\']}{src.suffix.lower()}"\n        shutil.copy2(src, dst)\n\n    command = [\n        sys.executable,\n        str(inference_script),\n        "-n",\n        args.model_name,\n        "-i",\n        str(input_dir),\n        "-o",\n        str(raw_dir),\n        "--outscale",\n        str(args.outscale),\n        "--suffix",\n        suffix,\n        "--tile",\n        str(args.tile),\n        "--ext",\n        args.ext,\n    ]\n    if args.model_path:\n        command.extend(["--model_path", str(args.model_path.resolve())])\n    if args.gpu_id is not None:\n        command.extend(["--gpu-id", str(args.gpu_id)])\n    if args.fp32:\n        command.append("--fp32")\n    if args.face_enhance:\n        command.append("--face_enhance")\n\n    print("Running:", " ".join(command))\n    subprocess.run(command, cwd=realesrgan_dir, check=True)\n    normalize_outputs(raw_dir, out_dir, rows, suffix)\n    print(f"Wrote normalized Real-ESRGAN outputs to {out_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(PROJECT_DIR / 'scripts' / 'make_kaggle_submission.py').write_text('from __future__ import annotations\n\nimport argparse\nimport base64\nimport csv\nimport zlib\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\ntry:\n    import cv2\nexcept ImportError:\n    cv2 = None\n\nIMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"]\n\n\ndef encode_image_bgr(image: np.ndarray) -> bytes:\n    image_to_encode = image.astype(np.uint8).flatten()\n    image_to_encode = np.append(image_to_encode, -1)\n    count = 1\n    rle: list[int] = []\n\n    for index in range(1, image_to_encode.shape[0]):\n        if image_to_encode[index] == image_to_encode[index - 1]:\n            count += 1\n            if count > 255:\n                rle += [int(image_to_encode[index - 1]), 255]\n                count = 1\n        else:\n            rle += [int(image_to_encode[index - 1]), count]\n            count = 1\n\n    compressed = zlib.compress(bytes(rle), zlib.Z_BEST_COMPRESSION)\n    return base64.b64encode(compressed)\n\n\ndef find_image(images_dir: Path, filename: str) -> Path:\n    exact = images_dir / filename\n    if exact.exists():\n        return exact\n\n    stem = Path(filename).stem\n    for extension in IMAGE_EXTENSIONS:\n        candidate = images_dir / f"{stem}{extension}"\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f"Could not find an SR image for {filename} in {images_dir}")\n\n\ndef read_image_bgr(path: str | Path) -> np.ndarray:\n    if cv2 is not None:\n        image = cv2.imread(str(path), cv2.IMREAD_COLOR)\n        if image is None:\n            raise ValueError(f"Could not read image: {path}")\n        return image\n\n    image_rgb = np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8)\n    return image_rgb[..., ::-1]\n\n\ndef rows_from_sample(sample_submission: Path, images_dir: Path, id_column: str, filename_column: str) -> list[dict[str, str]]:\n    with sample_submission.open("r", newline="", encoding="utf-8") as handle:\n        reader = csv.DictReader(handle)\n        rows = []\n        for row in reader:\n            filename = row[filename_column]\n            rows.append(\n                {\n                    "id": row.get(id_column, str(len(rows))),\n                    "filename": filename,\n                    "image_path": find_image(images_dir, filename).as_posix(),\n                }\n            )\n    return rows\n\n\ndef rows_from_images(images_dir: Path) -> list[dict[str, str]]:\n    images = sorted(path for path in images_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)\n    return [\n        {\n            "id": str(index),\n            "filename": path.name,\n            "image_path": path.as_posix(),\n        }\n        for index, path in enumerate(images)\n    ]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Encode super-resolution outputs for the SRGD Kaggle submission format.")\n    parser.add_argument("--images-dir", type=Path, required=True, help="Folder containing generated HR/SR images.")\n    parser.add_argument("--out", type=Path, default=Path("submission.csv"), help="Output submission CSV.")\n    parser.add_argument("--sample-submission", type=Path, default=None, help="Optional Kaggle sample_submission.csv.")\n    parser.add_argument("--id-column", default="id", help="ID column name in sample_submission.csv.")\n    parser.add_argument("--filename-column", default="filename", help="Filename column name in sample_submission.csv.")\n    parser.add_argument("--rle-column", default="rle", help="Encoded image column name for the output CSV.")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    images_dir = args.images_dir.resolve()\n    rows = (\n        rows_from_sample(args.sample_submission, images_dir, args.id_column, args.filename_column)\n        if args.sample_submission\n        else rows_from_images(images_dir)\n    )\n\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    with args.out.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[args.id_column, args.filename_column, args.rle_column])\n        writer.writeheader()\n        for row in rows:\n            image = read_image_bgr(row["image_path"])\n            writer.writerow(\n                {\n                    args.id_column: row["id"],\n                    args.filename_column: row["filename"],\n                    args.rle_column: str(encode_image_bgr(image)),\n                }\n            )\n\n    print(f"Wrote {len(rows)} encoded predictions to {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Kaggle helper scripts are ready.')

In [ ]:
# Compatibility patch for newer torchvision builds where functional_tensor was removed.
import site

def patch_basicsr_torchvision_import():
    site_roots = []
    try:
        site_roots.extend(site.getsitepackages())
    except AttributeError:
        pass
    try:
        site_roots.append(site.getusersitepackages())
    except AttributeError:
        pass

    patched = []
    for root in site_roots:
        target = Path(root) / 'basicsr' / 'data' / 'degradations.py'
        if not target.exists():
            continue
        text = target.read_text(encoding='utf-8')
        old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        new = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if old in text:
            target.write_text(text.replace(old, new), encoding='utf-8')
            patched.append(str(target))
    return patched

patched = patch_basicsr_torchvision_import()
print('Patched BasicSR files:' if patched else 'No BasicSR patch needed.', patched)
run([sys.executable, '-c', 'import torch, basicsr, realesrgan; print("imports ok", torch.__version__)'])

## 2. Check T4 x2

Notebook 无法替你切换 Kaggle accelerator；这一格只负责确认当前环境实际拿到了几张 GPU。

In [ ]:
import torch

GPU_COUNT = torch.cuda.device_count()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', GPU_COUNT)
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index))

if GPU_COUNT == 0:
    raise RuntimeError('No GPU found. In Kaggle Settings, choose GPU T4 x2 or at least one GPU.')
if KAGGLE and GPU_COUNT < 2:
    print('Warning: Kaggle is not using T4 x2 right now. Fine-tuning will fall back to the available GPU count.')

USE_GPUS = min(2, GPU_COUNT)

## 3. Discover the Kaggle dataset layout

支持常见目录：`train/lr + train/hr`、`train/270p + train/1080p`、`lq + gt`。如果你的数据目录不是这种命名，直接在第一格填写 `LR_DIR_OVERRIDE` 和 `HR_DIR_OVERRIDE`。

In [ ]:
from pathlib import Path
import os

IMAGE_EXTENSIONS = {'.bmp', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}
NAME_PAIRS = [
    ('lr', 'hr'),
    ('lq', 'gt'),
    ('low', 'high'),
    ('270p', '1080p'),
]


def has_images(folder: Path) -> bool:
    if folder is None or not folder.exists():
        return False
    return any(path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS for path in folder.rglob('*'))


def common_root(first: Path, second: Path) -> Path:
    return Path(os.path.commonpath([str(first.resolve()), str(second.resolve())]))


def candidate_roots():
    roots = []
    if KAGGLE and INPUT_ROOT.exists():
        roots.extend(sorted(path for path in INPUT_ROOT.iterdir() if path.is_dir()))
    smoke = PROJECT_DIR / 'data' / 'smoke' / 'raw'
    if smoke.exists():
        roots.append(smoke)
    return roots


def find_pair_dirs():
    if LR_DIR_OVERRIDE and HR_DIR_OVERRIDE:
        lr_dir = Path(LR_DIR_OVERRIDE)
        hr_dir = Path(HR_DIR_OVERRIDE)
        data_root = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else common_root(lr_dir, hr_dir)
        return data_root, lr_dir, hr_dir, 'manual'

    for root in candidate_roots():
        all_dirs = [path for path in root.rglob('*') if path.is_dir()]
        all_dirs.append(root)
        by_parent_and_name = {(path.parent.resolve(), path.name.lower()): path for path in all_dirs}
        for lr_name, hr_name in NAME_PAIRS:
            for lr_dir in all_dirs:
                if lr_dir.name.lower() != lr_name or not has_images(lr_dir):
                    continue
                hr_dir = by_parent_and_name.get((lr_dir.parent.resolve(), hr_name))
                if hr_dir and has_images(hr_dir):
                    return common_root(lr_dir, hr_dir), lr_dir, hr_dir, f'{lr_name}->{hr_name}'
    raise FileNotFoundError(
        'Could not auto-detect LR/HR folders. Set LR_DIR_OVERRIDE and HR_DIR_OVERRIDE in the config cell.'
    )


def find_test_lr_dir():
    if TEST_LR_DIR_OVERRIDE:
        return Path(TEST_LR_DIR_OVERRIDE)
    for root in candidate_roots():
        all_dirs = [path for path in root.rglob('*') if path.is_dir()]
        preferred_names = {'lr', 'lq', '270p'}
        test_dirs = [path for path in all_dirs if 'test' in {part.lower() for part in path.parts}]
        for path in test_dirs:
            if path.name.lower() in preferred_names and has_images(path):
                return path
        for path in test_dirs:
            if has_images(path):
                return path
    return None


def find_sample_submission():
    if SAMPLE_SUBMISSION_OVERRIDE:
        return Path(SAMPLE_SUBMISSION_OVERRIDE)
    if not INPUT_ROOT.exists():
        return None
    matches = sorted(INPUT_ROOT.rglob('sample_submission*.csv'))
    return matches[0] if matches else None

DATA_ROOT, LR_DIR, HR_DIR, DETECTED_LAYOUT = find_pair_dirs()
TEST_LR_DIR = find_test_lr_dir()
SAMPLE_SUBMISSION = find_sample_submission()

print('Detected layout:', DETECTED_LAYOUT)
print('DATA_ROOT:', DATA_ROOT)
print('LR_DIR:', LR_DIR)
print('HR_DIR:', HR_DIR)
print('TEST_LR_DIR:', TEST_LR_DIR)
print('SAMPLE_SUBMISSION:', SAMPLE_SUBMISSION)

## 4. Build paired metadata

生成两份文件：

- `data/kaggle_pairs.csv` 给本项目的 baseline / metric 脚本使用
- `data/kaggle_meta_info_srgd_pair.txt` 给 Real-ESRGAN paired dataset 使用

In [ ]:
import pandas as pd

PAIRS_CSV = DATA_DIR / 'kaggle_pairs.csv'
META_INFO = DATA_DIR / 'kaggle_meta_info_srgd_pair.txt'

prepare_cmd = [
    sys.executable, PROJECT_DIR / 'scripts' / 'prepare_pairs.py',
    '--data-root', DATA_ROOT,
    '--lr-dir', LR_DIR,
    '--hr-dir', HR_DIR,
    '--out', PAIRS_CSV,
    '--meta-info', META_INFO,
]
if PAIR_LIMIT is not None:
    prepare_cmd += ['--limit', PAIR_LIMIT]

run(prepare_cmd, cwd=PROJECT_DIR)
pairs = pd.read_csv(PAIRS_CSV)
print('pairs:', len(pairs))
print(pairs['split'].value_counts(dropna=False))
display(pairs.head())

## 5. Bicubic baseline and metrics

先跑一个轻量 baseline。它能验证数据配对、输出目录和 PSNR/SSIM 评估是否正常。

In [ ]:
BICUBIC_DIR = OUTPUT_DIR / 'bicubic'
BICUBIC_METRICS = OUTPUT_DIR / 'bicubic_metrics.csv'
BICUBIC_SUMMARY = OUTPUT_DIR / 'bicubic_summary.json'

run([
    sys.executable, PROJECT_DIR / 'scripts' / 'run_bicubic.py',
    '--pairs', PAIRS_CSV,
    '--out-dir', BICUBIC_DIR,
    '--overwrite',
], cwd=PROJECT_DIR)

run([
    sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
    '--pairs', PAIRS_CSV,
    '--sr-dir', BICUBIC_DIR,
    '--out', BICUBIC_METRICS,
    '--summary', BICUBIC_SUMMARY,
    '--resize-sr',
], cwd=PROJECT_DIR)

print(BICUBIC_SUMMARY.read_text())

## 6. Download Real-ESRGAN pretrained weights

微调需要 generator 和 discriminator 权重；推理只需要 generator 权重。

In [ ]:
from urllib.request import urlretrieve

PRETRAIN_DIR = REAL_ESRGAN_DIR / 'experiments' / 'pretrained_models'
PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    'RealESRGAN_x4plus.pth': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    'RealESRGAN_x4plus_netD.pth': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.3/RealESRGAN_x4plus_netD.pth',
}

for filename, url in WEIGHTS.items():
    target = PRETRAIN_DIR / filename
    if target.exists():
        print('exists:', target)
        continue
    print('downloading:', url)
    urlretrieve(url, target)
    print('saved:', target)

## 7. Pretrained Real-ESRGAN inference

这一步跑官方预训练模型，作为比 bicubic 更强的 baseline。`TILE = 256` 对 T4 通常比较稳，OOM 时改成 `128`。

In [ ]:
REALESRGAN_PRETRAINED_DIR = OUTPUT_DIR / 'realesrgan_x4plus_pretrained'
REALESRGAN_METRICS = OUTPUT_DIR / 'realesrgan_x4plus_pretrained_metrics.csv'
REALESRGAN_SUMMARY = OUTPUT_DIR / 'realesrgan_x4plus_pretrained_summary.json'

if RUN_PRETRAINED_INFERENCE:
    infer_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'run_realesrgan.py',
        '--realesrgan-dir', REAL_ESRGAN_DIR,
        '--pairs', PAIRS_CSV,
        '--out-dir', REALESRGAN_PRETRAINED_DIR,
        '--model-name', MODEL_NAME,
        '--outscale', 4,
        '--tile', TILE,
        '--gpu-id', 0,
    ]
    if INFERENCE_LIMIT is not None:
        infer_cmd += ['--limit', INFERENCE_LIMIT]
    run(infer_cmd, cwd=PROJECT_DIR)

    eval_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
        '--pairs', PAIRS_CSV,
        '--sr-dir', REALESRGAN_PRETRAINED_DIR,
        '--out', REALESRGAN_METRICS,
        '--summary', REALESRGAN_SUMMARY,
        '--resize-sr',
    ]
    if INFERENCE_LIMIT is not None:
        eval_cmd += ['--limit', INFERENCE_LIMIT]
    run(eval_cmd, cwd=PROJECT_DIR)
    print(REALESRGAN_SUMMARY.read_text())
else:
    print('Skipped pretrained Real-ESRGAN inference.')

## 8. Visual comparison grid

输出 `LR | Bicubic | RealESRGAN | HR`，用于快速看模型是否真的在生成高分辨率图。

In [ ]:
from IPython.display import Image as IPyImage, display

GRID_DIR = OUTPUT_DIR / 'grids'
methods = [f'Bicubic={BICUBIC_DIR}']
if RUN_PRETRAINED_INFERENCE and REALESRGAN_PRETRAINED_DIR.exists():
    methods.append(f'RealESRGAN={REALESRGAN_PRETRAINED_DIR}')

run([
    sys.executable, PROJECT_DIR / 'scripts' / 'make_visual_grid.py',
    '--pairs', PAIRS_CSV,
    '--methods', *methods,
    '--out-dir', GRID_DIR,
    '--limit', GRID_LIMIT,
], cwd=PROJECT_DIR)

first_grid = sorted(GRID_DIR.glob('*_grid.png'))[0]
print(first_grid)
display(IPyImage(filename=str(first_grid)))

## 9. Generate the fine-tuning config

这里生成 Real-ESRGAN paired-data 微调配置。`num_gpu: auto` 会让 BasicSR/Real-ESRGAN 使用当前可见 GPU 数，下一格用 2 个进程启动 DDP。

In [ ]:
FINETUNE_CONFIG = CONFIG_DIR / 'kaggle_finetune_realesrgan_x4plus_pairdata.yml'

run([
    sys.executable, PROJECT_DIR / 'scripts' / 'write_realesrgan_config.py',
    '--data-root', DATA_ROOT,
    '--meta-info', META_INFO,
    '--out', FINETUNE_CONFIG,
    '--batch-size', BATCH_SIZE_PER_GPU,
    '--workers', NUM_WORKERS_PER_GPU,
    '--gt-size', GT_SIZE,
    '--total-iter', TOTAL_ITER,
    '--save-freq', SAVE_FREQ,
    '--pretrain-g', 'experiments/pretrained_models/RealESRGAN_x4plus.pth',
    '--pretrain-d', 'experiments/pretrained_models/RealESRGAN_x4plus_netD.pth',
], cwd=PROJECT_DIR)

print(FINETUNE_CONFIG)
print(FINETUNE_CONFIG.read_text()[:2500])

## 10. Fine-tune on T4 x2

这格会在 Kaggle 双 T4 上用 DDP 启动 Real-ESRGAN。默认 smoke 模式只跑 100 iter；全量实验请把第一格改成 `RUN_MODE = "full"` 并按显存调整 `BATCH_SIZE_PER_GPU`。

In [ ]:
if RUN_FINETUNE:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = ','.join(str(index) for index in range(USE_GPUS))
    env.setdefault('OMP_NUM_THREADS', '2')

    if USE_GPUS >= 2:
        train_cmd = [
            sys.executable, '-m', 'torch.distributed.launch',
            f'--nproc_per_node={USE_GPUS}',
            '--master_port=4321',
            'realesrgan/train.py',
            '-opt', FINETUNE_CONFIG,
            '--launcher', 'pytorch',
            '--auto_resume',
        ]
    else:
        train_cmd = [
            sys.executable, 'realesrgan/train.py',
            '-opt', FINETUNE_CONFIG,
            '--auto_resume',
        ]

    run(train_cmd, cwd=REAL_ESRGAN_DIR, env=env)
else:
    print('Skipped fine-tuning.')

## 11. Locate the latest fine-tuned generator checkpoint

Real-ESRGAN 通常会把 checkpoint 放在 `Real-ESRGAN/experiments/<config name>/models`。

In [ ]:
MODEL_DIR = REAL_ESRGAN_DIR / 'experiments' / 'finetune_RealESRGANx4plus_SRGD_pairdata' / 'models'
checkpoint_candidates = []
if MODEL_DIR.exists():
    checkpoint_candidates = sorted(MODEL_DIR.glob('net_g_*.pth'), key=lambda path: path.stat().st_mtime)

FINETUNED_G = checkpoint_candidates[-1] if checkpoint_candidates else None
print('MODEL_DIR:', MODEL_DIR)
print('FINETUNED_G:', FINETUNED_G)

## 12. Optional: inference with the fine-tuned checkpoint

如果微调已经保存了 `net_g_*.pth`，这里会用它重新跑一轮验证集/样本推理和指标。

In [ ]:
FINETUNED_DIR = OUTPUT_DIR / 'realesrgan_x4plus_finetuned'
FINETUNED_METRICS = OUTPUT_DIR / 'realesrgan_x4plus_finetuned_metrics.csv'
FINETUNED_SUMMARY = OUTPUT_DIR / 'realesrgan_x4plus_finetuned_summary.json'

if FINETUNED_G is not None:
    infer_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'run_realesrgan.py',
        '--realesrgan-dir', REAL_ESRGAN_DIR,
        '--pairs', PAIRS_CSV,
        '--out-dir', FINETUNED_DIR,
        '--model-name', MODEL_NAME,
        '--model-path', FINETUNED_G,
        '--outscale', 4,
        '--tile', TILE,
        '--gpu-id', 0,
    ]
    if INFERENCE_LIMIT is not None:
        infer_cmd += ['--limit', INFERENCE_LIMIT]
    run(infer_cmd, cwd=PROJECT_DIR)

    eval_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
        '--pairs', PAIRS_CSV,
        '--sr-dir', FINETUNED_DIR,
        '--out', FINETUNED_METRICS,
        '--summary', FINETUNED_SUMMARY,
        '--resize-sr',
    ]
    if INFERENCE_LIMIT is not None:
        eval_cmd += ['--limit', INFERENCE_LIMIT]
    run(eval_cmd, cwd=PROJECT_DIR)
    print(FINETUNED_SUMMARY.read_text())
else:
    print('No fine-tuned generator checkpoint found yet.')

## 13. Course Metric Summary

For the course project, compute PSNR, SSIM, and LPIPS on the same subset and display a compact comparison table.


In [ ]:
import json
import pandas as pd

COURSE_METRIC_LIMIT = INFERENCE_LIMIT or COURSE_EVAL_LIMIT
COURSE_METHODS = {
    'Bicubic baseline': BICUBIC_DIR,
}
if REALESRGAN_PRETRAINED_DIR.exists():
    COURSE_METHODS['Real-ESRGAN pretrained'] = REALESRGAN_PRETRAINED_DIR
if FINETUNED_DIR.exists():
    COURSE_METHODS['Real-ESRGAN finetuned'] = FINETUNED_DIR

metric_rows = []
for method_name, sr_dir in COURSE_METHODS.items():
    safe_name = method_name.lower().replace(' ', '_').replace('-', '').replace('/', '_')
    summary_path = OUTPUT_DIR / f'{safe_name}_course_summary.json'
    metrics_path = OUTPUT_DIR / f'{safe_name}_course_metrics.csv'
    run([
        sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
        '--pairs', PAIRS_CSV,
        '--sr-dir', sr_dir,
        '--out', metrics_path,
        '--summary', summary_path,
        '--resize-sr',
        '--limit', COURSE_METRIC_LIMIT,
        '--lpips',
    ], cwd=PROJECT_DIR)
    data = json.loads(summary_path.read_text())
    metric_rows.append({
        'Method': method_name,
        'Images': data['count'],
        'PSNR (higher is better)': round(data['psnr_mean'], 4),
        'SSIM (higher is better)': round(data['ssim_mean'], 4),
        'LPIPS (lower is better)': round(data['lpips_mean'], 4),
    })

metric_table = pd.DataFrame(metric_rows)
display(metric_table)
metric_table.to_csv(OUTPUT_DIR / 'course_metric_summary.csv', index=False)
print('Saved:', OUTPUT_DIR / 'course_metric_summary.csv')

## 14. Optional Competition Submission, Not Needed For Course

如果数据集中有 `test/lr` 或类似目录，这里会对 test LR 图片推理，并生成比赛要求的 `submission.csv`。

In [ ]:
TEST_OUTPUT_DIR = OUTPUT_DIR / 'test_realesrgan'
SUBMISSION_CSV = WORK_ROOT / 'submission.csv'

if RUN_TEST_SUBMISSION and TEST_LR_DIR is not None:
    checkpoint_for_submission = FINETUNED_G if FINETUNED_G is not None else None
    model_args = ['--model_path', str(checkpoint_for_submission)] if checkpoint_for_submission else []

    # Split test images over the available GPUs for faster inference.
    staged_root = OUTPUT_DIR / 'test_lr_chunks'
    if staged_root.exists():
        shutil.rmtree(staged_root)
    TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    test_images = sorted(path for path in TEST_LR_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    print('test images:', len(test_images))

    processes = []
    for gpu_index in range(USE_GPUS):
        chunk_dir = staged_root / f'gpu{gpu_index}'
        chunk_dir.mkdir(parents=True, exist_ok=True)
        for image_path in test_images[gpu_index::USE_GPUS]:
            target = chunk_dir / image_path.name
            if not target.exists():
                shutil.copy2(image_path, target)

        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(gpu_index)
        cmd = [
            sys.executable, 'inference_realesrgan.py',
            '-n', MODEL_NAME,
            '-i', chunk_dir,
            '-o', TEST_OUTPUT_DIR,
            '--outscale', '4',
            '--suffix', '',
            '--tile', str(TILE),
            '--ext', 'png',
            *model_args,
        ]
        print('$ ' + ' '.join(str(part) for part in cmd))
        processes.append(subprocess.Popen([str(part) for part in cmd], cwd=REAL_ESRGAN_DIR, env=env))

    for process in processes:
        if process.wait() != 0:
            raise RuntimeError('One of the Real-ESRGAN test inference workers failed.')

    submit_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'make_kaggle_submission.py',
        '--images-dir', TEST_OUTPUT_DIR,
        '--out', SUBMISSION_CSV,
    ]
    if SAMPLE_SUBMISSION is not None:
        submit_cmd += ['--sample-submission', SAMPLE_SUBMISSION]
    run(submit_cmd, cwd=PROJECT_DIR)
    print('Submission:', SUBMISSION_CSV)
else:
    print('Skipped submission generation. TEST_LR_DIR was not found or RUN_TEST_SUBMISSION=False.')